# 03 — Pneumonia Detection: Computer Vision

**Component 3 of 4 — RespiraAI Deployment Capstone**

Transfer-learning CNN (MobileNetV2 backbone) classifying chest X-rays as NORMAL or PNEUMONIA, trained on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) dataset (Kermany et al.) — ~5,863 real pediatric chest X-rays.

A **9-image real sample** (3 NORMAL, 6 PNEUMONIA, pulled from the same dataset lineage) ships alongside this notebook so you can smoke-test the whole pipeline — loading, preprocessing, a full forward pass — immediately, without waiting on the ~1.1 GB full download. That's not remotely enough data to *train* a real classifier on, and the notebook is explicit about which cells are the smoke test versus the real training run.

In [9]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


device: cpu


## Get the full dataset

Two options — pick whichever you have set up. Both are commented out so this cell is safe to run without either configured; the notebook falls back to the bundled sample folder below.

In [10]:
# --- Option 1: kagglehub (recommended -- no kaggle.json juggling for public datasets) ---
# import kagglehub
# dataset_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
# print("Downloaded to:", dataset_path)

# --- Option 2: classic Kaggle API (needs a kaggle.json token, or KAGGLE_API_TOKEN env var --
# see the security note from our earlier session: never hardcode this in a notebook you'll share) ---
# import kaggle
# kaggle.api.dataset_download_files("paultimothymooney/chest-xray-pneumonia", path="../data", unzip=True)


## Locate the data directory

Checks for the full downloaded dataset first (`chest_xray/train`, `.../val`, `.../test` — the structure Kaggle ships), and falls back to the small bundled `chest_xray_sample/` folder if the full dataset isn't present yet.

In [11]:
def find_xray_root():
    full_dataset_candidates = [
        os.path.join("data", "chest_xray"),
        os.path.join("day1_data", "data", "chest_xray"),
        os.path.join("..", "data", "chest_xray"),
        "chest_xray",
    ]
    for c in full_dataset_candidates:
        if os.path.isdir(os.path.join(c, "train")):
            return c, True  # (root, is_full_dataset)

    sample_candidates = [
        os.path.join("data", "chest_xray_sample"),
        os.path.join("day1_data", "data", "chest_xray_sample"),
        os.path.join("..", "data", "chest_xray_sample"),
        "chest_xray_sample",
    ]
    for c in sample_candidates:
        if os.path.isdir(c):
            return c, False

    raise FileNotFoundError(
        "Neither the full chest_xray/ dataset nor the bundled chest_xray_sample/ folder was found. "
        "Download the full dataset (see cell above) or make sure chest_xray_sample/ shipped with the repo."
    )

XRAY_ROOT, HAS_FULL_DATASET = find_xray_root()
print(f"Using: {XRAY_ROOT}  (full dataset: {HAS_FULL_DATASET})")


Using: day1_data\data\chest_xray  (full dataset: True)


## Transforms

224x224 to match what MobileNetV2's ImageNet-pretrained weights expect. X-rays are single-channel grayscale, but the pretrained backbone expects 3 channels, so we replicate the channel rather than retrain the first conv layer from scratch — cheaper, and works fine in practice for this kind of transfer learning. Normalization stats are the standard ImageNet mean/std the backbone was originally trained with.

In [12]:
IMG_SIZE = 224

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.3),
    T.RandomRotation(degrees=7),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


## Model: MobileNetV2 transfer learning

MobileNetV2 over a larger backbone (ResNet50, etc.) specifically because the final Docker image needs to stay small enough for free-tier hosting — this is a deployment-driven choice, not just a training-speed one. Freezing the pretrained feature extractor and only training a new classification head is standard transfer learning: with ~5,800 images (tiny by ImageNet standards), fine-tuning the whole backbone risks overfitting and definitely isn't necessary to get strong results on a binary task like this.

In [13]:
def build_model(pretrained=True):
    weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
    net = models.mobilenet_v2(weights=weights)
    for param in net.features.parameters():
        param.requires_grad = False  # freeze the pretrained backbone
    net.classifier[1] = nn.Linear(net.last_channel, 1)  # new head, binary logit, trainable
    return net.to(DEVICE)

cnn_model = build_model(pretrained=HAS_FULL_DATASET)
# pretrained=False when only the bundled sample is available and this environment has no
# internet access to fetch ImageNet weights (e.g. a locked-down sandbox) -- swap to True
# once you're running this with the full dataset and normal internet access, for real results.
print(cnn_model.classifier)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\moham/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:04<00:00, 3.46MB/s]

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)


## Smoke test: pipeline mechanics on the real bundled sample

This runs regardless of whether the full dataset is present — it confirms image loading, transforms, batching, and the forward pass all work correctly on real chest X-rays before you invest time in a full training run. **This is not training or evaluation** — 9 images is nowhere near enough for either; it's a pipeline correctness check.

In [14]:
sample_root = XRAY_ROOT if not HAS_FULL_DATASET else next(
    (
        p
        for p in [
            os.path.join("data", "chest_xray_sample"),
            os.path.join("day1_data", "data", "chest_xray_sample"),
            os.path.join("..", "data", "chest_xray_sample"),
        ]
        if os.path.isdir(p)
    ),
    None,
)

if sample_root:
    smoke_dataset = ImageFolder(sample_root, transform=transform)
    print("classes:", smoke_dataset.classes)
    print("images:", len(smoke_dataset))

    smoke_loader = DataLoader(smoke_dataset, batch_size=4, shuffle=True)
    imgs, labels = next(iter(smoke_loader))
    print("batch shape:", imgs.shape, " labels:", labels.tolist())

    cnn_model.eval()
    with torch.no_grad():
        logits = cnn_model(imgs.to(DEVICE))
        probs = torch.sigmoid(logits).cpu().squeeze()
    print("predicted PNEUMONIA probabilities (untrained head, so these are near-random):", probs.tolist())
    print("\nPipeline confirmed working end-to-end on real chest X-ray images.")
else:
    print("No sample folder found to smoke-test against.")


classes: ['NORMAL', 'PNEUMONIA']
images: 9
batch shape: torch.Size([4, 3, 224, 224])  labels: [1, 0, 1, 1]
predicted PNEUMONIA probabilities (untrained head, so these are near-random): [0.4204446077346802, 0.4867808222770691, 0.45065367221832275, 0.46243807673454285]

Pipeline confirmed working end-to-end on real chest X-ray images.


## Full training (run once you have the complete dataset)

Everything below is written to run against the real ~5,800-image dataset once downloaded via the cell near the top. Guarded by `HAS_FULL_DATASET` so it's safe to run this whole notebook top-to-bottom even before you've downloaded anything — it'll just skip straight past this section.

In [15]:
def evaluate(net, loader):
    net.eval()
    correct, total, all_preds, all_labels = 0, 0, [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.float().to(DEVICE)
            logits = net(imgs).squeeze(1)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return correct / total, all_preds, all_labels


if HAS_FULL_DATASET:
    from sklearn.metrics import classification_report

    train_ds = ImageFolder(os.path.join(XRAY_ROOT, "train"), transform=train_transform)
    val_ds = ImageFolder(os.path.join(XRAY_ROOT, "val"), transform=transform)
    test_ds = ImageFolder(os.path.join(XRAY_ROOT, "test"), transform=transform)

    print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}  classes={train_ds.classes}")

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

    # class imbalance: the real dataset skews toward PNEUMONIA -- same idea as notebooks 1/2
    n_normal = sum(1 for _, label in train_ds.samples if label == train_ds.class_to_idx["NORMAL"])
    n_pneumonia = len(train_ds) - n_normal
    pos_weight = torch.tensor([n_normal / n_pneumonia]).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(cnn_model.classifier.parameters(), lr=1e-3)  # only the head is trainable

    EPOCHS = 5  # the frozen backbone + small trainable head converges fast; raise if needed
    for epoch in range(EPOCHS):
        cnn_model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.float().to(DEVICE).unsqueeze(1)
            optimizer.zero_grad()
            logits = cnn_model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        val_acc, _, _ = evaluate(cnn_model, val_loader)
        print(f"epoch {epoch+1}/{EPOCHS}  train_loss={running_loss/len(train_ds):.4f}  val_acc={val_acc:.3f}")

    test_acc, test_preds, test_labels = evaluate(cnn_model, test_loader)
    print(f"\nTest accuracy: {test_acc:.3f}")
    print(classification_report(test_labels, test_preds, target_names=test_ds.classes))
else:
    print("Skipping full training -- full dataset not found. Run the download cell near the top first.")


train=5216  val=16  test=624  classes=['NORMAL', 'PNEUMONIA']
epoch 1/5  train_loss=0.2283  val_acc=0.875
epoch 2/5  train_loss=0.1425  val_acc=0.938
epoch 3/5  train_loss=0.1268  val_acc=1.000
epoch 4/5  train_loss=0.1089  val_acc=0.938
epoch 5/5  train_loss=0.1081  val_acc=1.000

Test accuracy: 0.873
              precision    recall  f1-score   support

      NORMAL       0.82      0.84      0.83       234
   PNEUMONIA       0.90      0.89      0.90       390

    accuracy                           0.87       624
   macro avg       0.86      0.87      0.87       624
weighted avg       0.87      0.87      0.87       624



## Save the model

In [17]:
MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

if HAS_FULL_DATASET:
    torch.save(cnn_model.state_dict(), os.path.join(MODELS_DIR, "xray_cnn.pt"))
    print(f"Saved trained weights to {MODELS_DIR}/xray_cnn.pt")
else:
    print("Not saving -- this model only has an untrained head (smoke-test only), "
          "not a real trained model. Download the full dataset and re-run before saving.")


Saved trained weights to models/xray_cnn.pt


## Notes

- The smoke test above is a real, useful step in its own right: it catches transform bugs, channel-count mismatches, and DataLoader issues *before* you burn time on a full training run that then fails at epoch 1 for a dumb preprocessing reason.
- `pos_weight` shows up here too, same idea as Notebook 2 — the real dataset also skews toward the PNEUMONIA class, so the same imbalance-handling pattern applies across every component of this project, not just the tabular one.
- Freezing the backbone and training only the classifier head is both a training-time and a deployment-time decision: fewer trainable parameters, faster convergence, and (via MobileNetV2) a smaller final model to ship in the Docker image.
- Next: `04_rag_pipeline.ipynb` — a completely different kind of component, text retrieval + an LLM call, using the same `openai/gpt-oss-20b` setup from the earlier prompt-engineering lab.